In [0]:
# Databricks notebook source
# ══════════════════════════════════════
# ANALYSIS — physical_lojas
# Squad 3 — Arquitetura Medalhao
# Objetivo: cruzar KPIs Gold e gerar
#           insights de negocio
# Regra: le apenas tabelas Gold
#        nao recalcula, nao grava
# ══════════════════════════════════════

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# tabelas Gold — unica fonte de dados

GOLD_KPI4_TABLE  = f"{TARGET_SCHEMA}.gold_kpi_receita_loja_ano"
GOLD_KPI5_TABLE  = f"{TARGET_SCHEMA}.gold_kpi_crescimento_yoy_loja"
GOLD_KPI6L_TABLE = f"{TARGET_SCHEMA}.gold_kpi_transacoes_loja_mes"
GOLD_KPI8L_TABLE = f"{TARGET_SCHEMA}.gold_kpi_ticket_medio_semestre"
GOLD_KPI7L_TABLE = f"{TARGET_SCHEMA}.gold_kpi_lojas_enriquecimento_ibge"

print("Tabelas Gold configuradas!")

In [0]:
# ler todas as tabelas Gold

from pyspark.sql.functions import (
    col, count, sum as spark_sum,
    avg, round as spark_round,
    max as spark_max, min as spark_min,
    when, desc, asc, abs as spark_abs
)
from pyspark.sql.types import DoubleType, IntegerType

df_kpi4 = read_sql_table(spark, GOLD_KPI4_TABLE) \
    .withColumn("receita_total_ano",
        col("receita_total_ano").cast(DoubleType())) \
    .withColumn("total_transacoes",
        col("total_transacoes").cast(IntegerType())) \
    .withColumn("ranking_receita",
        col("ranking_receita").cast(IntegerType())) \
    .withColumn("ano",
        col("ano").cast(IntegerType())) \
    .withColumn("peso_vendas",
        col("peso_vendas").cast(IntegerType()))

df_kpi5 = read_sql_table(spark, GOLD_KPI5_TABLE) \
    .withColumn("receita_total_ano",
        col("receita_total_ano").cast(DoubleType())) \
    .withColumn("receita_ano_anterior",
        col("receita_ano_anterior").cast(DoubleType())) \
    .withColumn("crescimento_yoy_pct",
        col("crescimento_yoy_pct").cast(DoubleType())) \
    .withColumn("ano",
        col("ano").cast(IntegerType()))

df_kpi6l = read_sql_table(spark, GOLD_KPI6L_TABLE) \
    .withColumn("total_transacoes",
        col("total_transacoes").cast(IntegerType())) \
    .withColumn("ano",
        col("ano").cast(IntegerType())) \
    .withColumn("mes",
        col("mes").cast(IntegerType()))

df_kpi8l = read_sql_table(spark, GOLD_KPI8L_TABLE) \
    .withColumn("ticket_medio_valor",
        col("ticket_medio_valor").cast(DoubleType())) \
    .withColumn("ticket_medio_itens",
        col("ticket_medio_itens").cast(DoubleType())) \
    .withColumn("total_transacoes",
        col("total_transacoes").cast(IntegerType())) \
    .withColumn("ano",
        col("ano").cast(IntegerType())) \
    .withColumn("semestre",
        col("semestre").cast(IntegerType()))

print("Gold KPIs carregados!")
print(f"   KPI 4 (receita/ano)        : {df_kpi4.count():,} linhas")
print(f"   KPI 5 (YoY)                : {df_kpi5.count():,} linhas")
print(f"   KPI 6L (transacoes/mes)    : {df_kpi6l.count():,} linhas")
print(f"   KPI 8L (ticket/semestre)   : {df_kpi8l.count():,} linhas")

In [0]:
# ══════════════════════════════════════
# INSIGHT 1
# Lojas com maior receita tem maior
# volume de transacoes ou maior ticket?
# ══════════════════════════════════════

print("=" * 55)
print("INSIGHT 1 — RECEITA VEM DE VOLUME OU TICKET?")
print("=" * 55)

df_receita_anual = df_kpi4.select(
    "id_loja", "nome_loja", "estado_loja",
    "ano", "receita_total_ano",
    "total_transacoes", "ranking_receita"
)

df_ticket_anual = df_kpi8l \
    .groupBy("id_loja", "ano") \
    .agg(
        spark_round(avg(col("ticket_medio_valor")), 2).alias("ticket_medio_ano"),
        spark_round(avg(col("ticket_medio_itens")), 2).alias("itens_medio_ano"),
    )

df_insight1 = df_receita_anual \
    .join(df_ticket_anual, on=["id_loja", "ano"], how="left") \
    .withColumn(
        "perfil_loja",
        when(
            (col("total_transacoes") > 10000) &
            (col("ticket_medio_ano") > 50),
            "Alto Volume + Alto Ticket"
        )
        .when(
            (col("total_transacoes") > 10000) &
            (col("ticket_medio_ano") <= 50),
            "Alto Volume + Baixo Ticket"
        )
        .when(
            (col("total_transacoes") <= 10000) &
            (col("ticket_medio_ano") > 50),
            "Baixo Volume + Alto Ticket"
        )
        .otherwise("Baixo Volume + Baixo Ticket")
    )

print("\nPerfil das lojas — Volume vs Ticket:")
display(
    df_insight1
    .select(
        "ranking_receita",
        "nome_loja",
        "estado_loja",
        "ano",
        "receita_total_ano",
        "total_transacoes",
        "ticket_medio_ano",
        "itens_medio_ano",
        "perfil_loja"
    )
    .orderBy("ano", "ranking_receita")
)

print("\nDistribuicao de perfis:")
display(
    df_insight1
    .groupBy("perfil_loja")
    .agg(count("id_loja").alias("total_lojas"))
    .orderBy(desc("total_lojas"))
)

In [0]:
# ══════════════════════════════════════
# INSIGHT 2
# Lojas que crescem em receita (YoY)
# tambem crescem em volume de transacoes?
# ══════════════════════════════════════

print("=" * 55)
print("INSIGHT 2 — CRESCIMENTO YoY VEM DE RECEITA OU VOLUME?")
print("=" * 55)

df_transacoes_anuais = df_kpi6l \
    .groupBy("id_loja", "ano") \
    .agg(spark_sum("total_transacoes").alias("total_transacoes_ano"))

df_insight2 = df_kpi5 \
    .filter(col("crescimento_yoy_pct").isNotNull()) \
    .join(df_transacoes_anuais, on=["id_loja", "ano"], how="left") \
    .withColumn(
        "crescimento_receita",
        when(col("crescimento_yoy_pct") > 0, "Cresceu")
        .when(col("crescimento_yoy_pct") < 0, "Caiu")
        .otherwise("Estavel")
    )

print("\nCrescimento YoY — Receita vs Transacoes:")
display(
    df_insight2
    .select(
        "nome_loja",
        "estado_loja",
        "ano",
        "crescimento_yoy_pct",
        "crescimento_receita",
        "total_transacoes_ano",
        "receita_total_ano",
    )
    .orderBy(desc("crescimento_yoy_pct"))
)

print("\nResumo por tipo de crescimento:")
display(
    df_insight2
    .groupBy("crescimento_receita")
    .agg(
        count("id_loja").alias("total_lojas"),
        spark_round(avg("crescimento_yoy_pct"), 2).alias("media_crescimento_pct"),
        spark_round(avg("total_transacoes_ano"), 0).alias("media_transacoes"),
    )
    .orderBy(desc("media_crescimento_pct"))
)

In [0]:
# ══════════════════════════════════════
# INSIGHT 3
# Lojas com maior peso de vendas
# realmente performam melhor?
# ══════════════════════════════════════

print("=" * 55)
print("INSIGHT 3 — PESO DE VENDAS VS PERFORMANCE REAL")
print("=" * 55)

df_insight3 = df_kpi4 \
    .join(
        df_kpi8l
        .groupBy("id_loja", "ano")
        .agg(
            spark_round(avg("ticket_medio_valor"), 2).alias("ticket_medio"),
            spark_sum("total_transacoes").alias("total_transacoes_semestres"),
        ),
        on=["id_loja", "ano"],
        how="left"
    ) \
    .withColumn(
        "porte_loja",
        when(col("peso_vendas") >= 8, "Grande (8-10)")
        .when(col("peso_vendas") >= 5, "Medio (5-7)")
        .otherwise("Pequena (1-4)")
    )

print("\nPerformance media por porte de loja:")
display(
    df_insight3
    .groupBy("porte_loja")
    .agg(
        count("id_loja").alias("total_lojas"),
        spark_round(avg("receita_total_ano"), 2).alias("receita_media_ano"),
        spark_round(avg("ticket_medio"), 2).alias("ticket_medio"),
        spark_round(avg("total_transacoes"), 0).alias("media_transacoes"),
    )
    .orderBy(desc("receita_media_ano"))
)

print("\nLojas que superam a media do seu porte:")
media_por_porte = df_insight3 \
    .groupBy("porte_loja") \
    .agg(avg("receita_total_ano").alias("media_porte"))

display(
    df_insight3
    .join(media_por_porte, on="porte_loja", how="left")
    .filter(col("receita_total_ano") > col("media_porte"))
    .select(
        "nome_loja",
        "estado_loja",
        "porte_loja",
        "peso_vendas",
        "receita_total_ano",
        spark_round(
            col("receita_total_ano") - col("media_porte"), 2
        ).alias("acima_da_media")
    )
    .orderBy(desc("acima_da_media"))
)

In [0]:
# ══════════════════════════════════════
# INSIGHT 4
# Existe concentracao de receita?
# Poucos estados/lojas geram a maior
# parte da receita?
# ══════════════════════════════════════

print("=" * 55)
print("INSIGHT 4 — CONCENTRACAO DE RECEITA")
print("=" * 55)

receita_total_geral = df_kpi4 \
    .agg(spark_sum("receita_total_ano")) \
    .collect()[0][0]

df_insight4 = df_kpi4 \
    .groupBy("estado_loja") \
    .agg(
        spark_round(
            spark_sum("receita_total_ano"), 2
        ).alias("receita_estado"),
        count("id_loja").alias("total_lojas"),
    ) \
    .withColumn(
        "participacao_pct",
        spark_round(
            col("receita_estado") / receita_total_geral * 100, 2
        )
    ) \
    .orderBy(desc("receita_estado"))

print(f"\nReceita total geral: R$ {receita_total_geral:,.2f}")
print("\nParticipacao por estado:")
display(df_insight4)

print("\nTop 3 estados concentram:")
top3_participacao = df_insight4 \
    .limit(3) \
    .agg(spark_sum("participacao_pct")) \
    .collect()[0][0]
print(f"   {top3_participacao:.1f}% da receita total")

In [0]:
# ══════════════════════════════════════
# INSIGHT 5 — Enriquecimento IBGE
# Lojas em regioes mais populosas
# tem maior receita?
# ══════════════════════════════════════

print("=" * 55)
print("INSIGHT 5 — REGIOES GEOGRAFICAS vs RECEITA")
print("=" * 55)

try:
    df_kpi7l = read_sql_table(spark, GOLD_KPI7L_TABLE)

    df_insight5 = df_kpi4 \
        .join(
            df_kpi7l.select(
                "id_loja",
                "nome_regiao",
                "nome_mesorregiao"
            ),
            on="id_loja",
            how="left"
        )

    print("\nReceita media por regiao geografica:")
    display(
        df_insight5
        .groupBy("nome_regiao")
        .agg(
            count("id_loja").alias("total_lojas"),
            spark_round(
                avg("receita_total_ano"), 2
            ).alias("receita_media_ano"),
            spark_round(
                spark_sum("receita_total_ano"), 2
            ).alias("receita_total_regiao"),
        )
        .orderBy(desc("receita_media_ano"))
    )

    print("\nReceita media por mesorregiao:")
    display(
        df_insight5
        .filter(col("nome_mesorregiao").isNotNull())
        .groupBy("nome_regiao", "nome_mesorregiao")
        .agg(
            count("id_loja").alias("total_lojas"),
            spark_round(
                avg("receita_total_ano"), 2
            ).alias("receita_media_ano"),
        )
        .orderBy(desc("receita_media_ano"))
        .limit(10)
    )

except Exception as e:
    print(f"Atencao: KPI IBGE nao disponivel. {e}")

In [0]:
# ══════════════════════════════════════
# RESUMO EXECUTIVO
# ══════════════════════════════════════

print("=" * 55)
print("RESUMO EXECUTIVO — LOJAS FISICAS")
print("=" * 55)

total_lojas      = df_kpi4.select("id_loja").distinct().count()
total_estados    = df_kpi4.select("estado_loja").distinct().count()
receita_total    = df_kpi4 \
    .agg(spark_round(spark_sum("receita_total_ano"), 2)) \
    .collect()[0][0]
receita_media    = df_kpi4 \
    .agg(spark_round(avg("receita_total_ano"), 2)) \
    .collect()[0][0]
loja_top1        = df_kpi4 \
    .filter(col("ranking_receita") == 1) \
    .select("nome_loja", "receita_total_ano") \
    .first()
ticket_medio     = df_kpi8l \
    .agg(spark_round(avg("ticket_medio_valor"), 2)) \
    .collect()[0][0]
total_transacoes = df_kpi6l \
    .agg(spark_sum("total_transacoes")) \
    .collect()[0][0]

# lojas que cresceram YoY
cresceram = df_kpi5 \
    .filter(
        col("crescimento_yoy_pct").isNotNull() &
        (col("crescimento_yoy_pct") > 0)
    ).count()

total_com_yoy = df_kpi5 \
    .filter(col("crescimento_yoy_pct").isNotNull()) \
    .count()

print(f"""
Visao Geral:
   Total de lojas ativas     : {total_lojas:,}
   Estados presentes         : {total_estados:,}

Performance:
   Receita total             : R$ {receita_total:,.2f}
   Receita media por loja    : R$ {receita_media:,.2f}
   Ticket medio              : R$ {ticket_medio:,.2f}
   Total de transacoes       : {total_transacoes:,}

Destaques:
   Loja top 1                : {loja_top1['nome_loja']}
   Receita loja top 1        : R$ {loja_top1['receita_total_ano']:,.2f}
   Lojas com crescimento YoY : {cresceram}/{total_com_yoy}

Insights principais:
   INSIGHT 1 : Perfil receita = volume vs ticket
   INSIGHT 2 : Crescimento YoY vs volume transacoes
   INSIGHT 3 : Peso de vendas vs performance real
   INSIGHT 4 : Concentracao de receita por estado
   INSIGHT 5 : Regioes geograficas vs receita (IBGE)
""")